# Exercise 1: Back-Propagation in a Feed-Forward Neural Network

This notebook works through **two full iterations** of forward propagation and backpropagation on a feed-forward network with **two hidden layers**. Every intermediate value is printed: each weighted sum, each activation, each error term (δ), each gradient, and each weight update.

The arithmetic is done neuron by neuron in plain Python so every multiplication is visible. At the end, a vectorized NumPy version and a numerical gradient check confirm the hand calculations.

## 1. Network Architecture and Initial State

| Component | Choice |
|---|---|
| **Input layer** | 3 neurons: $x = [0.37,\ -0.82,\ 0.55]$ |
| **Hidden layer 1** | 3 neurons, sigmoid activation |
| **Hidden layer 2** | 2 neurons, sigmoid activation |
| **Output layer** | 2 neurons, sigmoid activation |
| **Target output** | $y = [0.23,\ 0.91]$ |
| **Activation function** | Sigmoid: $\sigma(z) = \dfrac{1}{1+e^{-z}}$, with derivative $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$ |
| **Loss function** | Mean squared error (half sum of squares): $L = \tfrac{1}{2}\sum_k (\hat{y}_k - y_k)^2$ |
| **Learning rate** | $\eta = 0.35$ |
| **Training** | One training example, two gradient-descent steps |

```
 Input (3)        Hidden 1 (3)        Hidden 2 (2)        Output (2)
   x1 ─┐         ┌─ h1_1 ─┐          ┌─ h2_1 ─┐          ┌─ o1  → ŷ1
   x2 ─┼─ W1,b1 ─┼─ h1_2 ─┼─ W2,b2 ──┤        ├─ W3,b3 ──┤
   x3 ─┘         └─ h1_3 ─┘          └─ h2_2 ─┘          └─ o2  → ŷ2
```

**Notation.** $w^{(l)}_{ji}$ is the weight from neuron $i$ in layer $l-1$ to neuron $j$ in layer $l$. So $W^{(l)}$ has one **row per neuron in layer $l$** and one **column per neuron in the layer before it**.

$$z^{(l)}_j = \sum_i w^{(l)}_{ji}\, a^{(l-1)}_i + b^{(l)}_j, \qquad a^{(l)}_j = \sigma\!\left(z^{(l)}_j\right), \qquad a^{(0)} = x$$

### Initial weights and biases (chosen by hand)

$$
W^{(1)} = \begin{bmatrix} 0.27 & -0.41 & 0.63 \\ -0.58 & 0.19 & 0.44 \\ 0.71 & -0.36 & -0.22 \end{bmatrix},\quad
b^{(1)} = \begin{bmatrix} 0.13 \\ -0.29 \\ 0.07 \end{bmatrix}
$$

$$
W^{(2)} = \begin{bmatrix} 0.52 & -0.67 & 0.31 \\ -0.24 & 0.48 & 0.86 \end{bmatrix},\quad
b^{(2)} = \begin{bmatrix} -0.17 \\ 0.21 \end{bmatrix}
$$

$$
W^{(3)} = \begin{bmatrix} 0.93 & -0.38 \\ -0.46 & 0.74 \end{bmatrix},\quad
b^{(3)} = \begin{bmatrix} 0.09 \\ -0.12 \end{bmatrix}
$$

In [20]:
import math
import copy

# ----- Data -----
x = [0.37, -0.82, 0.55]      # input vector
y = [0.23, 0.91]             # target vector
eta = 0.35                   # learning rate

# ----- Initial parameters -----
# W[l][j][i] = weight from neuron i (layer l-1) to neuron j (layer l)
initial_params = {
    "W1": [[ 0.27, -0.41,  0.63],
           [-0.58,  0.19,  0.44],
           [ 0.71, -0.36, -0.22]],
    "b1": [0.13, -0.29, 0.07],
    "W2": [[ 0.52, -0.67,  0.31],
           [-0.24,  0.48,  0.86]],
    "b2": [-0.17, 0.21],
    "W3": [[ 0.93, -0.38],
           [-0.46,  0.74]],
    "b3": [0.09, -0.12],
}
params = copy.deepcopy(initial_params)

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

def f(v):
    # Format a number to 6 decimals for display (full precision is kept in memory).
    return f"{v:.6f}"

def p(v):
    # Format a number in parentheses for use inside an expression.
    return f"({v:.6f})" if v < 0 else f"{v:.6f}"

def show_params(params):
    for name in ["W1", "b1", "W2", "b2", "W3", "b3"]:
        val = params[name]
        if isinstance(val[0], list):
            print(f"{name} =")
            for row in val:
                print("   [" + ", ".join(f"{v:10.6f}" for v in row) + " ]")
        else:
            print(f"{name} = [" + ", ".join(f"{v:10.6f}" for v in val) + " ]")
    print()

print("Initial parameters:\n")
show_params(params)

Initial parameters:

W1 =
   [  0.270000,  -0.410000,   0.630000 ]
   [ -0.580000,   0.190000,   0.440000 ]
   [  0.710000,  -0.360000,  -0.220000 ]
b1 = [  0.130000,  -0.290000,   0.070000 ]
W2 =
   [  0.520000,  -0.670000,   0.310000 ]
   [ -0.240000,   0.480000,   0.860000 ]
b2 = [ -0.170000,   0.210000 ]
W3 =
   [  0.930000,  -0.380000 ]
   [ -0.460000,   0.740000 ]
b3 = [  0.090000,  -0.120000 ]



In [21]:
# ---------- Step functions used in both iterations ----------

def forward_layer(W, b, a_prev, layer, prev_name):
    # Compute z = W·a_prev + b and a = sigmoid(z) for one layer, printing every term.
    z_list, a_list = [], []
    for j in range(len(W)):
        terms = " + ".join(f"{p(W[j][i])}·{p(a_prev[i])}" for i in range(len(a_prev)))
        z = sum(W[j][i] * a_prev[i] for i in range(len(a_prev))) + b[j]
        a = sigmoid(z)
        print(f"z{layer}_{j+1} = {terms} + {p(b[j])}")
        print(f"      = {f(z)}")
        print(f"a{layer}_{j+1} = σ({f(z)}) = 1 / (1 + e^({f(-z)})) = {f(a)}\n")
        z_list.append(z)
        a_list.append(a)
    return z_list, a_list

def forward(params, x):
    print("── Hidden layer 1 (inputs: x) ──")
    z1, a1 = forward_layer(params["W1"], params["b1"], x, 1, "x")
    print("── Hidden layer 2 (inputs: a1) ──")
    z2, a2 = forward_layer(params["W2"], params["b2"], a1, 2, "a1")
    print("── Output layer (inputs: a2) ──")
    z3, a3 = forward_layer(params["W3"], params["b3"], a2, 3, "a2")
    print("Network output ŷ = [" + ", ".join(f(v) for v in a3) + "]")
    return {"z1": z1, "a1": a1, "z2": z2, "a2": a2, "z3": z3, "a3": a3}

def loss(y_hat, y):
    parts = [(y_hat[k] - y[k]) ** 2 for k in range(len(y))]
    for k in range(len(y)):
        print(f"(ŷ{k+1} − y{k+1})² = ({f(y_hat[k])} − {f(y[k])})² = ({f(y_hat[k]-y[k])})² = {f(parts[k])}")
    L = 0.5 * sum(parts)
    print(f"\nL = ½ · ({' + '.join(f(v) for v in parts)}) = {f(L)}")
    return L

def output_deltas(y_hat, y):
    # δ3_k = ∂L/∂z3_k = (ŷ_k − y_k) · ŷ_k(1 − ŷ_k)
    d = []
    for k in range(len(y)):
        dL_da = y_hat[k] - y[k]
        da_dz = y_hat[k] * (1 - y_hat[k])
        d.append(dL_da * da_dz)
        print(f"δ3_{k+1} = (ŷ{k+1} − y{k+1}) · ŷ{k+1}(1 − ŷ{k+1})")
        print(f"     = ({f(y_hat[k])} − {f(y[k])}) · {f(y_hat[k])}·(1 − {f(y_hat[k])})")
        print(f"     = {p(dL_da)} · {f(da_dz)} = {f(d[-1])}\n")
    return d

def hidden_deltas(W_next, d_next, a, layer):
    # δl_j = (Σ_k w(l+1)_kj · δ(l+1)_k) · a_j(1 − a_j)   — uses the weights BEFORE updating
    d = []
    for j in range(len(a)):
        back = sum(W_next[k][j] * d_next[k] for k in range(len(d_next)))
        terms = " + ".join(f"{p(W_next[k][j])}·{p(d_next[k])}" for k in range(len(d_next)))
        da_dz = a[j] * (1 - a[j])
        d.append(back * da_dz)
        print(f"δ{layer}_{j+1} = (Σ_k w{layer+1}_k{j+1} · δ{layer+1}_k) · a{layer}_{j+1}(1 − a{layer}_{j+1})")
        print(f"     = ({terms}) · {f(a[j])}·(1 − {f(a[j])})")
        print(f"     = {p(back)} · {f(da_dz)} = {f(d[-1])}\n")
    return d

def gradients(d, a_prev, layer, prev_name):
    # ∂L/∂w_ji = δ_j · a_prev_i ,  ∂L/∂b_j = δ_j
    gW, gb = [], []
    for j in range(len(d)):
        row = []
        for i in range(len(a_prev)):
            g = d[j] * a_prev[i]
            row.append(g)
            print(f"∂L/∂w{layer}_{j+1}{i+1} = δ{layer}_{j+1} · {prev_name}_{i+1} = {p(d[j])} · {p(a_prev[i])} = {f(g)}")
        gW.append(row)
        gb.append(d[j])
        print(f"∂L/∂b{layer}_{j+1}  = δ{layer}_{j+1} = {f(d[j])}\n")
    return gW, gb

def update(params, grads, eta):
    # w_new = w_old − η · ∂L/∂w
    new = copy.deepcopy(params)
    for l in (1, 2, 3):
        W, b = params[f"W{l}"], params[f"b{l}"]
        gW, gb = grads[f"W{l}"], grads[f"b{l}"]
        print(f"── Layer {l} ──")
        for j in range(len(W)):
            for i in range(len(W[j])):
                new[f"W{l}"][j][i] = W[j][i] - eta * gW[j][i]
                print(f"w{l}_{j+1}{i+1}: {p(W[j][i])} − {eta}·{p(gW[j][i])} = {f(new[f'W{l}'][j][i])}")
            new[f"b{l}"][j] = b[j] - eta * gb[j]
            print(f"b{l}_{j+1} : {p(b[j])} − {eta}·{p(gb[j])} = {f(new[f'b{l}'][j])}")
        print()
    return new

### Backpropagation formulas used

By the chain rule, define the error term of each neuron as $\delta^{(l)}_j = \dfrac{\partial L}{\partial z^{(l)}_j}$.

**Output layer** (sigmoid + half-MSE):
$$\delta^{(3)}_k = \frac{\partial L}{\partial \hat y_k}\cdot\frac{\partial \hat y_k}{\partial z^{(3)}_k} = (\hat y_k - y_k)\,\hat y_k(1-\hat y_k)$$

**Hidden layers** (error flows back through the *pre-update* weights):
$$\delta^{(l)}_j = \Big(\sum_k w^{(l+1)}_{kj}\,\delta^{(l+1)}_k\Big)\, a^{(l)}_j\,(1-a^{(l)}_j)$$

**Gradients**:
$$\frac{\partial L}{\partial w^{(l)}_{ji}} = \delta^{(l)}_j\, a^{(l-1)}_i, \qquad \frac{\partial L}{\partial b^{(l)}_j} = \delta^{(l)}_j$$

**Gradient-descent update**:
$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

All δ values and gradients are computed first, and only then are the weights updated.

---
## 2. Iteration 1

### Step 1.1: Forward propagation

In [22]:
print("ITERATION 1: FORWARD PASS\n")
cache1 = forward(params, x)

ITERATION 1: FORWARD PASS

── Hidden layer 1 (inputs: x) ──
z1_1 = 0.270000·0.370000 + (-0.410000)·(-0.820000) + 0.630000·0.550000 + 0.130000
      = 0.912600
a1_1 = σ(0.912600) = 1 / (1 + e^(-0.912600)) = 0.713532

z1_2 = (-0.580000)·0.370000 + 0.190000·(-0.820000) + 0.440000·0.550000 + (-0.290000)
      = -0.418400
a1_2 = σ(-0.418400) = 1 / (1 + e^(0.418400)) = 0.396900

z1_3 = 0.710000·0.370000 + (-0.360000)·(-0.820000) + (-0.220000)·0.550000 + 0.070000
      = 0.506900
a1_3 = σ(0.506900) = 1 / (1 + e^(-0.506900)) = 0.624079

── Hidden layer 2 (inputs: a1) ──
z2_1 = 0.520000·0.713532 + (-0.670000)·0.396900 + 0.310000·0.624079 + (-0.170000)
      = 0.128578
a2_1 = σ(0.128578) = 1 / (1 + e^(-0.128578)) = 0.532100

z2_2 = (-0.240000)·0.713532 + 0.480000·0.396900 + 0.860000·0.624079 + 0.210000
      = 0.765973
a2_2 = σ(0.765973) = 1 / (1 + e^(-0.765973)) = 0.682649

── Output layer (inputs: a2) ──
z3_1 = 0.930000·0.532100 + (-0.380000)·0.682649 + 0.090000
      = 0.325447
a3_1 = σ(0.325

### Step 1.2: Loss

In [23]:
L1 = loss(cache1["a3"], y)

(ŷ1 − y1)² = (0.580651 − 0.230000)² = (0.350651)² = 0.122956
(ŷ2 − y2)² = (0.535041 − 0.910000)² = (-0.374959)² = 0.140594

L = ½ · (0.122956 + 0.140594) = 0.131775


### Step 1.3: Backpropagation, output layer error terms δ⁽³⁾

In [24]:
d3_1 = output_deltas(cache1["a3"], y)

δ3_1 = (ŷ1 − y1) · ŷ1(1 − ŷ1)
     = (0.580651 − 0.230000) · 0.580651·(1 − 0.580651)
     = 0.350651 · 0.243495 = 0.085382

δ3_2 = (ŷ2 − y2) · ŷ2(1 − ŷ2)
     = (0.535041 − 0.910000) · 0.535041·(1 − 0.535041)
     = (-0.374959) · 0.248772 = -0.093279



### Step 1.4: Backpropagation, hidden layer 2 error terms δ⁽²⁾

In [25]:
d2_1 = hidden_deltas(params["W3"], d3_1, cache1["a2"], 2)

δ2_1 = (Σ_k w3_k1 · δ3_k) · a2_1(1 − a2_1)
     = (0.930000·0.085382 + (-0.460000)·(-0.093279)) · 0.532100·(1 − 0.532100)
     = 0.122314 · 0.248970 = 0.030452

δ2_2 = (Σ_k w3_k2 · δ3_k) · a2_2(1 − a2_2)
     = ((-0.380000)·0.085382 + 0.740000·(-0.093279)) · 0.682649·(1 − 0.682649)
     = (-0.101472) · 0.216639 = -0.021983



### Step 1.5: Backpropagation, hidden layer 1 error terms δ⁽¹⁾

In [26]:
d1_1 = hidden_deltas(params["W2"], d2_1, cache1["a1"], 1)

δ1_1 = (Σ_k w2_k1 · δ2_k) · a1_1(1 − a1_1)
     = (0.520000·0.030452 + (-0.240000)·(-0.021983)) · 0.713532·(1 − 0.713532)
     = 0.021111 · 0.204404 = 0.004315

δ1_2 = (Σ_k w2_k2 · δ2_k) · a1_2(1 − a1_2)
     = ((-0.670000)·0.030452 + 0.480000·(-0.021983)) · 0.396900·(1 − 0.396900)
     = (-0.030955) · 0.239370 = -0.007410

δ1_3 = (Σ_k w2_k3 · δ2_k) · a1_3(1 − a1_3)
     = (0.310000·0.030452 + 0.860000·(-0.021983)) · 0.624079·(1 − 0.624079)
     = (-0.009465) · 0.234604 = -0.002221



### Step 1.6: Gradients for every weight and bias

In [27]:
grads1 = {}
print("── Output layer (W3, b3) ──")
grads1["W3"], grads1["b3"] = gradients(d3_1, cache1["a2"], 3, "a2")
print("── Hidden layer 2 (W2, b2) ──")
grads1["W2"], grads1["b2"] = gradients(d2_1, cache1["a1"], 2, "a1")
print("── Hidden layer 1 (W1, b1) ──")
grads1["W1"], grads1["b1"] = gradients(d1_1, x, 1, "x")

── Output layer (W3, b3) ──
∂L/∂w3_11 = δ3_1 · a2_1 = 0.085382 · 0.532100 = 0.045432
∂L/∂w3_12 = δ3_1 · a2_2 = 0.085382 · 0.682649 = 0.058286
∂L/∂b3_1  = δ3_1 = 0.085382

∂L/∂w3_21 = δ3_2 · a2_1 = (-0.093279) · 0.532100 = -0.049634
∂L/∂w3_22 = δ3_2 · a2_2 = (-0.093279) · 0.682649 = -0.063677
∂L/∂b3_2  = δ3_2 = -0.093279

── Hidden layer 2 (W2, b2) ──
∂L/∂w2_11 = δ2_1 · a1_1 = 0.030452 · 0.713532 = 0.021729
∂L/∂w2_12 = δ2_1 · a1_2 = 0.030452 · 0.396900 = 0.012087
∂L/∂w2_13 = δ2_1 · a1_3 = 0.030452 · 0.624079 = 0.019005
∂L/∂b2_1  = δ2_1 = 0.030452

∂L/∂w2_21 = δ2_2 · a1_1 = (-0.021983) · 0.713532 = -0.015685
∂L/∂w2_22 = δ2_2 · a1_2 = (-0.021983) · 0.396900 = -0.008725
∂L/∂w2_23 = δ2_2 · a1_3 = (-0.021983) · 0.624079 = -0.013719
∂L/∂b2_2  = δ2_2 = -0.021983

── Hidden layer 1 (W1, b1) ──
∂L/∂w1_11 = δ1_1 · x_1 = 0.004315 · 0.370000 = 0.001597
∂L/∂w1_12 = δ1_1 · x_2 = 0.004315 · (-0.820000) = -0.003538
∂L/∂w1_13 = δ1_1 · x_3 = 0.004315 · 0.550000 = 0.002373
∂L/∂b1_1  = δ1_1 = 0.004315

∂L/

### Step 1.7: Update weights and biases (η = 0.35)

In [28]:
params_before_1 = copy.deepcopy(params)
params = update(params, grads1, eta)
print("Parameters after iteration 1:\n")
show_params(params)

── Layer 1 ──
w1_11: 0.270000 − 0.35·0.001597 = 0.269441
w1_12: (-0.410000) − 0.35·(-0.003538) = -0.408762
w1_13: 0.630000 − 0.35·0.002373 = 0.629169
b1_1 : 0.130000 − 0.35·0.004315 = 0.128490
w1_21: (-0.580000) − 0.35·(-0.002742) = -0.579040
w1_22: 0.190000 − 0.35·0.006076 = 0.187873
w1_23: 0.440000 − 0.35·(-0.004075) = 0.441426
b1_2 : (-0.290000) − 0.35·(-0.007410) = -0.287407
w1_31: 0.710000 − 0.35·(-0.000822) = 0.710288
w1_32: (-0.360000) − 0.35·0.001821 = -0.360637
w1_33: (-0.220000) − 0.35·(-0.001221) = -0.219573
b1_3 : 0.070000 − 0.35·(-0.002221) = 0.070777

── Layer 2 ──
w2_11: 0.520000 − 0.35·0.021729 = 0.512395
w2_12: (-0.670000) − 0.35·0.012087 = -0.674230
w2_13: 0.310000 − 0.35·0.019005 = 0.303348
b2_1 : (-0.170000) − 0.35·0.030452 = -0.180658
w2_21: (-0.240000) − 0.35·(-0.015685) = -0.234510
w2_22: 0.480000 − 0.35·(-0.008725) = 0.483054
w2_23: 0.860000 − 0.35·(-0.013719) = 0.864802
b2_2 : 0.210000 − 0.35·(-0.021983) = 0.217694

── Layer 3 ──
w3_11: 0.930000 − 0.35·0.045432

---
## 3. Iteration 2

### Step 2.1: Forward propagation

In [29]:
print("ITERATION 2: FORWARD PASS\n")
cache2 = forward(params, x)

ITERATION 2: FORWARD PASS

── Hidden layer 1 (inputs: x) ──
z1_1 = 0.269441·0.370000 + (-0.408762)·(-0.820000) + 0.629169·0.550000 + 0.128490
      = 0.909411
a1_1 = σ(0.909411) = 1 / (1 + e^(-0.909411)) = 0.712880

z1_2 = (-0.579040)·0.370000 + 0.187873·(-0.820000) + 0.441426·0.550000 + (-0.287407)
      = -0.412923
a1_2 = σ(-0.412923) = 1 / (1 + e^(0.412923)) = 0.398211

z1_3 = 0.710288·0.370000 + (-0.360637)·(-0.820000) + (-0.219573)·0.550000 + 0.070777
      = 0.508541
a1_3 = σ(0.508541) = 1 / (1 + e^(-0.508541)) = 0.624464

── Hidden layer 2 (inputs: a1) ──
z2_1 = 0.512395·0.712880 + (-0.674230)·0.398211 + 0.303348·0.624464 + (-0.180658)
      = 0.105562
a2_1 = σ(0.105562) = 1 / (1 + e^(-0.105562)) = 0.526366

z2_2 = (-0.234510)·0.712880 + 0.483054·0.398211 + 0.864802·0.624464 + 0.217694
      = 0.782912
a2_2 = σ(0.782912) = 1 / (1 + e^(-0.782912)) = 0.686307

── Output layer (inputs: a2) ──
z3_1 = 0.914099·0.526366 + (-0.400400)·0.686307 + 0.060116
      = 0.266469
a3_1 = σ(0.266

### Step 2.2: Loss

In [30]:
L2 = loss(cache2["a3"], y)

(ŷ1 − y1)² = (0.566226 − 0.230000)² = (0.336226)² = 0.113048
(ŷ2 − y2)² = (0.550534 − 0.910000)² = (-0.359466)² = 0.129216

L = ½ · (0.113048 + 0.129216) = 0.121132


### Step 2.3: Backpropagation, output layer error terms δ⁽³⁾

In [31]:
d3_2 = output_deltas(cache2["a3"], y)

δ3_1 = (ŷ1 − y1) · ŷ1(1 − ŷ1)
     = (0.566226 − 0.230000) · 0.566226·(1 − 0.566226)
     = 0.336226 · 0.245614 = 0.082582

δ3_2 = (ŷ2 − y2) · ŷ2(1 − ŷ2)
     = (0.550534 − 0.910000) · 0.550534·(1 − 0.550534)
     = (-0.359466) · 0.247446 = -0.088949



### Step 2.4: Backpropagation, hidden layer 2 error terms δ⁽²⁾

In [32]:
d2_2 = hidden_deltas(params["W3"], d3_2, cache2["a2"], 2)

δ2_1 = (Σ_k w3_k1 · δ3_k) · a2_1(1 − a2_1)
     = (0.914099·0.082582 + (-0.442628)·(-0.088949)) · 0.526366·(1 − 0.526366)
     = 0.114859 · 0.249305 = 0.028635

δ2_2 = (Σ_k w3_k2 · δ3_k) · a2_2(1 − a2_2)
     = ((-0.400400)·0.082582 + 0.762287·(-0.088949)) · 0.686307·(1 − 0.686307)
     = (-0.100870) · 0.215290 = -0.021716



### Step 2.5: Backpropagation, hidden layer 1 error terms δ⁽¹⁾

In [33]:
d1_2 = hidden_deltas(params["W2"], d2_2, cache2["a1"], 1)

δ1_1 = (Σ_k w2_k1 · δ2_k) · a1_1(1 − a1_1)
     = (0.512395·0.028635 + (-0.234510)·(-0.021716)) · 0.712880·(1 − 0.712880)
     = 0.019765 · 0.204682 = 0.004046

δ1_2 = (Σ_k w2_k2 · δ2_k) · a1_2(1 − a1_2)
     = ((-0.674230)·0.028635 + 0.483054·(-0.021716)) · 0.398211·(1 − 0.398211)
     = (-0.029797) · 0.239639 = -0.007140

δ1_3 = (Σ_k w2_k3 · δ2_k) · a1_3(1 − a1_3)
     = (0.303348·0.028635 + 0.864802·(-0.021716)) · 0.624464·(1 − 0.624464)
     = (-0.010094) · 0.234509 = -0.002367



### Step 2.6: Gradients for every weight and bias

In [34]:
grads2 = {}
print("── Output layer (W3, b3) ──")
grads2["W3"], grads2["b3"] = gradients(d3_2, cache2["a2"], 3, "a2")
print("── Hidden layer 2 (W2, b2) ──")
grads2["W2"], grads2["b2"] = gradients(d2_2, cache2["a1"], 2, "a1")
print("── Hidden layer 1 (W1, b1) ──")
grads2["W1"], grads2["b1"] = gradients(d1_2, x, 1, "x")

── Output layer (W3, b3) ──
∂L/∂w3_11 = δ3_1 · a2_1 = 0.082582 · 0.526366 = 0.043468
∂L/∂w3_12 = δ3_1 · a2_2 = 0.082582 · 0.686307 = 0.056677
∂L/∂b3_1  = δ3_1 = 0.082582

∂L/∂w3_21 = δ3_2 · a2_1 = (-0.088949) · 0.526366 = -0.046820
∂L/∂w3_22 = δ3_2 · a2_2 = (-0.088949) · 0.686307 = -0.061046
∂L/∂b3_2  = δ3_2 = -0.088949

── Hidden layer 2 (W2, b2) ──
∂L/∂w2_11 = δ2_1 · a1_1 = 0.028635 · 0.712880 = 0.020413
∂L/∂w2_12 = δ2_1 · a1_2 = 0.028635 · 0.398211 = 0.011403
∂L/∂w2_13 = δ2_1 · a1_3 = 0.028635 · 0.624464 = 0.017882
∂L/∂b2_1  = δ2_1 = 0.028635

∂L/∂w2_21 = δ2_2 · a1_1 = (-0.021716) · 0.712880 = -0.015481
∂L/∂w2_22 = δ2_2 · a1_2 = (-0.021716) · 0.398211 = -0.008648
∂L/∂w2_23 = δ2_2 · a1_3 = (-0.021716) · 0.624464 = -0.013561
∂L/∂b2_2  = δ2_2 = -0.021716

── Hidden layer 1 (W1, b1) ──
∂L/∂w1_11 = δ1_1 · x_1 = 0.004046 · 0.370000 = 0.001497
∂L/∂w1_12 = δ1_1 · x_2 = 0.004046 · (-0.820000) = -0.003317
∂L/∂w1_13 = δ1_1 · x_3 = 0.004046 · 0.550000 = 0.002225
∂L/∂b1_1  = δ1_1 = 0.004046

∂L/

### Step 2.7: Update weights and biases (η = 0.35)

In [35]:
params_before_2 = copy.deepcopy(params)
params = update(params, grads2, eta)
print("Parameters after iteration 2:\n")
show_params(params)

── Layer 1 ──
w1_11: 0.269441 − 0.35·0.001497 = 0.268917
w1_12: (-0.408762) − 0.35·(-0.003317) = -0.407600
w1_13: 0.629169 − 0.35·0.002225 = 0.628391
b1_1 : 0.128490 − 0.35·0.004046 = 0.127074
w1_21: (-0.579040) − 0.35·(-0.002642) = -0.578116
w1_22: 0.187873 − 0.35·0.005855 = 0.185824
w1_23: 0.441426 − 0.35·(-0.003927) = 0.442801
b1_2 : (-0.287407) − 0.35·(-0.007140) = -0.284907
w1_31: 0.710288 − 0.35·(-0.000876) = 0.710594
w1_32: (-0.360637) − 0.35·0.001941 = -0.361317
w1_33: (-0.219573) − 0.35·(-0.001302) = -0.219117
b1_3 : 0.070777 − 0.35·(-0.002367) = 0.071606

── Layer 2 ──
w2_11: 0.512395 − 0.35·0.020413 = 0.505250
w2_12: (-0.674230) − 0.35·0.011403 = -0.678221
w2_13: 0.303348 − 0.35·0.017882 = 0.297090
b2_1 : (-0.180658) − 0.35·0.028635 = -0.190681
w2_21: (-0.234510) − 0.35·(-0.015481) = -0.229092
w2_22: 0.483054 − 0.35·(-0.008648) = 0.486080
w2_23: 0.864802 − 0.35·(-0.013561) = 0.869548
b2_2 : 0.217694 − 0.35·(-0.021716) = 0.225295

── Layer 3 ──
w3_11: 0.914099 − 0.35·0.043468

---
## 4. Result after two iterations

One more forward pass with the twice-updated weights shows whether training is moving the output toward the target.

In [36]:
print("FORWARD PASS WITH PARAMETERS AFTER ITERATION 2\n")
cache3 = forward(params, x)
print()
L3 = loss(cache3["a3"], y)

print("\n================ SUMMARY ================")
print(f"{'':22}{'ŷ1':>10}{'ŷ2':>10}{'Loss':>12}")
print(f"{'Target':22}{y[0]:>10.6f}{y[1]:>10.6f}")
print(f"{'Iteration 1 (initial)':22}{cache1['a3'][0]:>10.6f}{cache1['a3'][1]:>10.6f}{L1:>12.6f}")
print(f"{'Iteration 2':22}{cache2['a3'][0]:>10.6f}{cache2['a3'][1]:>10.6f}{L2:>12.6f}")
print(f"{'After 2 updates':22}{cache3['a3'][0]:>10.6f}{cache3['a3'][1]:>10.6f}{L3:>12.6f}")
print(f"\nLoss reduced by {L1 - L3:.6f} ({100*(L1 - L3)/L1:.2f}%) over two iterations.")

FORWARD PASS WITH PARAMETERS AFTER ITERATION 2

── Hidden layer 1 (inputs: x) ──
z1_1 = 0.268917·0.370000 + (-0.407600)·(-0.820000) + 0.628391·0.550000 + 0.127074
      = 0.906420
a1_1 = σ(0.906420) = 1 / (1 + e^(-0.906420)) = 0.712267

z1_2 = (-0.578116)·0.370000 + 0.185824·(-0.820000) + 0.442801·0.550000 + (-0.284907)
      = -0.407646
a1_2 = σ(-0.407646) = 1 / (1 + e^(0.407646)) = 0.399477

z1_3 = 0.710594·0.370000 + (-0.361317)·(-0.820000) + (-0.219117)·0.550000 + 0.071606
      = 0.510291
a1_3 = σ(0.510291) = 1 / (1 + e^(-0.510291)) = 0.624875

── Hidden layer 2 (inputs: a1) ──
z2_1 = 0.505250·0.712267 + (-0.678221)·0.399477 + 0.297090·0.624875 + (-0.190681)
      = 0.083903
a2_1 = σ(0.083903) = 1 / (1 + e^(-0.083903)) = 0.520963

z2_2 = (-0.229092)·0.712267 + 0.486080·0.399477 + 0.869548·0.624875 + 0.225295
      = 0.799657
a2_2 = σ(0.799657) = 1 / (1 + e^(-0.799657)) = 0.689901

── Output layer (inputs: a2) ──
z3_1 = 0.898885·0.520963 + (-0.420237)·0.689901 + 0.031213
      = 0.

---
## 5. Verification

### 5.1 Vectorized NumPy implementation
The same two iterations in matrix form ($z = Wa + b$, $\delta^{(l)} = (W^{(l+1)})^\top \delta^{(l+1)} \odot a^{(l)}(1-a^{(l)})$, $\nabla W = \delta\, a_{\text{prev}}^\top$). The results should match the step-by-step values above.

In [37]:
import numpy as np

def np_step(P, x, y, eta):
    x = np.array(x); y = np.array(y)
    sig = lambda z: 1 / (1 + np.exp(-z))
    a1 = sig(P["W1"] @ x + P["b1"])
    a2 = sig(P["W2"] @ a1 + P["b2"])
    a3 = sig(P["W3"] @ a2 + P["b3"])
    L = 0.5 * np.sum((a3 - y) ** 2)
    d3 = (a3 - y) * a3 * (1 - a3)
    d2 = (P["W3"].T @ d3) * a2 * (1 - a2)
    d1 = (P["W2"].T @ d2) * a1 * (1 - a1)
    G = {"W3": np.outer(d3, a2), "b3": d3,
         "W2": np.outer(d2, a1), "b2": d2,
         "W1": np.outer(d1, x),  "b1": d1}
    newP = {k: P[k] - eta * G[k] for k in P}
    return newP, L, a3, G

P = {k: np.array(v, dtype=float) for k, v in initial_params.items()}
P, L1_np, out1, G1_np = np_step(P, x, y, eta)
P, L2_np, out2, G2_np = np_step(P, x, y, eta)

print(f"Iteration 1 loss: NumPy {L1_np:.6f} | step-by-step {L1:.6f}")
print(f"Iteration 2 loss: NumPy {L2_np:.6f} | step-by-step {L2:.6f}")
max_diff = max(np.max(np.abs(P[k] - np.array(params[k]))) for k in P)
print(f"Largest difference between final parameters: {max_diff:.2e}")

Iteration 1 loss: NumPy 0.131775 | step-by-step 0.131775
Iteration 2 loss: NumPy 0.121132 | step-by-step 0.121132
Largest difference between final parameters: 0.00e+00


### 5.2 Numerical gradient check (iteration 1)
Each analytic gradient from backpropagation is compared with a central finite difference:
$\dfrac{\partial L}{\partial w} \approx \dfrac{L(w+\epsilon) - L(w-\epsilon)}{2\epsilon}$, with $\epsilon = 10^{-5}$.

In [38]:
def np_loss(P, x, y):
    sig = lambda z: 1 / (1 + np.exp(-z))
    a1 = sig(P["W1"] @ np.array(x) + P["b1"])
    a2 = sig(P["W2"] @ a1 + P["b2"])
    a3 = sig(P["W3"] @ a2 + P["b3"])
    return 0.5 * np.sum((a3 - np.array(y)) ** 2)

P0 = {k: np.array(v, dtype=float) for k, v in initial_params.items()}
eps = 1e-5
worst = 0.0
for k in ["W1", "b1", "W2", "b2", "W3", "b3"]:
    num = np.zeros_like(P0[k])
    for idx in np.ndindex(P0[k].shape):
        Pp = {kk: vv.copy() for kk, vv in P0.items()}; Pp[k][idx] += eps
        Pm = {kk: vv.copy() for kk, vv in P0.items()}; Pm[k][idx] -= eps
        num[idx] = (np_loss(Pp, x, y) - np_loss(Pm, x, y)) / (2 * eps)
    diff = np.max(np.abs(num - np.array(grads1[k])))
    worst = max(worst, diff)
    print(f"{k}: max |analytic − numerical| = {diff:.2e}")
print("\nGradient check passed." if worst < 1e-8 else "\nGradient check FAILED.")

W1: max |analytic − numerical| = 3.21e-12
b1: max |analytic − numerical| = 4.81e-13
W2: max |analytic − numerical| = 2.61e-12
b2: max |analytic − numerical| = 1.72e-12
W3: max |analytic − numerical| = 1.19e-12
b3: max |analytic − numerical| = 3.20e-12

Gradient check passed.


---
## 6. Conclusion

- The network was set up with 3 inputs, two sigmoid hidden layers (3 and 2 neurons), 2 sigmoid outputs, half-MSE loss, and a learning rate of 0.35.
- In each iteration, the forward pass computed every weighted sum and activation, and the loss compared the output with the target $y = [0.23, 0.91]$.
- Backpropagation applied the chain rule from the output layer back to hidden layer 1, computing $\delta$ for each neuron with the pre-update weights, then a gradient for each of the 19 weights and 7 biases.
- After two gradient-descent updates the loss went down (see the summary table in Section 4), so the outputs moved toward the targets.
- The NumPy version gives the same results, and the finite-difference check confirms the analytic gradients.